In [1]:
# %%
"""
Single-cell Jupyter version – outputs KML using TIFF valid-pixel contours
and computes overlap using actual patch polygons.

Fixes vs. original:
  1. Buffer(0) on all reprojected polygons to fix topology errors.
  2. Sliver filter: discard intersection polygons thinner than a threshold.
  3. Use intersection-over-min-area (not IoU) — more meaningful for patches.
  4. Raised default threshold to something sensible (0.5%).
  5. Added an absolute area guard: ignore intersections < min_overlap_m2.
"""

# --- USER CONFIG -------------------------------------------------------
root_dir = "/home/ubuntu/SENSERO/GeoTiff/Patch_336/"  # ← adjust
overlap_threshold = 0.0001        # percent of *smaller* patch that must overlap
min_overlap_m2    = 0.0     # ignore intersections smaller than this (m²)

# --- IMPORTS -----------------------------------------------------------
import os
import csv
import random
import math
from collections import namedtuple
import rasterio
from rasterio.warp import transform_bounds, transform_geom
from rasterio.features import shapes as rio_shapes
from shapely.geometry import shape, Polygon, MultiPolygon, mapping
from shapely.ops import unary_union
from shapely.validation import make_valid
import warnings

try:
    import simplekml
except ModuleNotFoundError:
    simplekml = None

Bounds = namedtuple("Bounds", ["left", "bottom", "right", "top"])

# Approximate degrees-to-metres at Romanian latitudes (~45°N)
DEG_TO_M_LON = math.cos(math.radians(45)) * 111_320  # ~78 km per degree lon
DEG_TO_M_LAT = 111_320                                # ~111 km per degree lat

def approx_area_m2(polygon):
    """Rough area in m² for a WGS84 polygon at ~45°N latitude."""
    # Shapely .area gives deg², convert
    return polygon.area * DEG_TO_M_LON * DEG_TO_M_LAT

# --- HELPERS -----------------------------------------------------------

def get_bounds(filepath, target_crs="EPSG:4326"):
    """Return bbox (left,bottom,right,top) in WGS-84."""
    with rasterio.open(filepath) as src:
        b = src.bounds
        if src.crs is not None:
            b = transform_bounds(
                src.crs, target_crs,
                b.left, b.bottom, b.right, b.top,
                densify_pts=21,
            )
    return Bounds(*b)


def get_valid_polygon(filepath, target_crs="EPSG:4326"):
    """
    Return a single (Multi)Polygon of valid (non-nodata) pixels in WGS84.
    Applies buffer(0) to fix topology issues from reprojection.
    """
    with rasterio.open(filepath) as src:
        mask = src.read_masks(1)  # 255 = valid, 0 = nodata
        geom_parts = []
        for geom, val in rio_shapes(mask, mask=mask, transform=src.transform):
            if val == 0:
                continue
            if src.crs is not None and src.crs.to_string() != target_crs:
                geom = transform_geom(src.crs, target_crs, geom, precision=8)
            try:
                s = shape(geom)
                s = make_valid(s.buffer(0))
                if not s.is_empty:
                    geom_parts.append(s)
            except Exception as e:
                print(f"  [WARN] polygon fix failed for {filepath}: {e}")
    if not geom_parts:
        return None
    union = unary_union(geom_parts)
    union = make_valid(union.buffer(0))
    return union


def boxes_overlap(b1, b2):
    """Fast AABB rejection test."""
    return not (
        b1.right <= b2.left or b1.left >= b2.right or
        b1.top   <= b2.bottom or b1.bottom >= b2.top
    )


def centre(b):
    return ((b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0)


def patch_is_3360m(filepath, tolerance=5):
    """Returns True if patch is 3360×3360 m (±tolerance)."""
    with rasterio.open(filepath) as src:
        if not src.crs or src.crs.is_geographic:
            return True
        resx, resy = src.res
        width_m  = abs(resx) * src.width
        height_m = abs(resy) * src.height
        return (
            abs(width_m  - 3360) <= tolerance and
            abs(height_m - 3360) <= tolerance
        )


def is_sliver(polygon, ratio_threshold=0.05):
    """
    Detect sliver polygons: very small area relative to perimeter.
    A circle has area/perimeter² ≈ 0.0796; a sliver → 0.
    """
    if polygon.is_empty:
        return True
    if polygon.length == 0:
        return True
    compactness = polygon.area / (polygon.length ** 2)
    return compactness < ratio_threshold * 0.01  # very conservative


def compute_overlap_pct(poly1, poly2):
    """
    Overlap as percentage of the *smaller* patch's area.
    Returns 0 if the intersection is a sliver or below min_overlap_m2.
    """
    if poly1 is None or poly2 is None:
        return 0.0
    try:
        if not poly1.intersects(poly2):
            return 0.0

        inter = poly1.intersection(poly2)
        inter = make_valid(inter.buffer(0))

        if inter.is_empty:
            return 0.0

        # Filter out line/point intersections – keep only polygonal parts
        if inter.geom_type == "GeometryCollection":
            polys = [g for g in inter.geoms
                     if isinstance(g, (Polygon, MultiPolygon))]
            if not polys:
                return 0.0
            inter = unary_union(polys)

        if not isinstance(inter, (Polygon, MultiPolygon)):
            return 0.0

        # Sliver check
        sub_polys = [inter] if isinstance(inter, Polygon) else list(inter.geoms)
        real_parts = [p for p in sub_polys if not is_sliver(p)]
        if not real_parts:
            return 0.0
        inter = unary_union(real_parts)

        # Absolute-area guard (approximate)
        area_m2 = approx_area_m2(inter)
        if area_m2 < min_overlap_m2:
            return 0.0

        # Percentage of the smaller patch
        min_area = min(poly1.area, poly2.area)
        if min_area == 0:
            return 0.0
        return (inter.area / min_area) * 100.0

    except Exception as e:
        print(f"  [WARN] overlap computation failed: {e}")
        return 0.0


# --- FILE DISCOVERY ----------------------------------------------------

def find_geotiff_files(root_dir):
    """Discover GeoTIFF patches inside Multispectral folders."""
    out_bounds = {}
    out_shapes = {}

    for dirpath, dirnames, filenames in os.walk(root_dir):
        dirnames[:] = [d for d in dirnames if d != ".ipynb_checkpoints"]
        parts = [p.lower() for p in dirpath.split(os.sep)]
        if "multispectral" not in parts:
            continue

        for fname in filenames:
            if not fname.lower().endswith((".tif", ".tiff")):
                continue
            tif = os.path.join(dirpath, fname)
            try:
                if not patch_is_3360m(tif):
                    print(f"[INFO ] Skipping {tif} (not 3360×3360 m)")
                    continue
                out_bounds[tif] = get_bounds(tif)
                poly = get_valid_polygon(tif)
                if poly is not None:
                    out_shapes[tif] = poly
                else:
                    print(f"[WARN ] No valid polygon for {tif}")
            except Exception as e:
                print(f"[ERROR] reading {tif}: {e}")

    return out_bounds, out_shapes


# --- KML OUTPUT --------------------------------------------------------

def write_kml(overlap_pairs, shapes_dict, kml_path):
    """KML with actual valid-data contours and centre points."""
    if simplekml is None:
        print("[WARN ] simplekml not installed – skipping KML")
        return
    kml = simplekml.Kml()
    done = set()
    for f1, f2, _ in overlap_pairs:
        for fname in (f1, f2):
            if fname in done:
                continue
            done.add(fname)
            base = os.path.basename(fname)
            shp = shapes_dict.get(fname)
            if shp is None:
                continue
            polys = [shp] if isinstance(shp, Polygon) else list(shp.geoms)
            for idx, poly in enumerate(polys):
                try:
                    coords = [(x, y) for x, y in poly.exterior.coords]
                    kp = kml.newpolygon(
                        name=f"{base} ({idx})",
                        outerboundaryis=coords,
                    )
                    kp.style.polystyle.color  = "3300ff00"
                    kp.style.linestyle.color  = "ff0000ff"
                    kp.style.linestyle.width  = 1
                    c = poly.centroid
                    pt = kml.newpoint(
                        name=f"{base} centre",
                        coords=[(c.x, c.y)],
                    )
                    pt.style.iconstyle.color = "ff0000ff"
                    pt.style.iconstyle.scale = 1.0
                except Exception as e:
                    print(f"  [WARN] KML polygon for {base}: {e}")
    kml.save(kml_path)
    print(f"KML saved → {kml_path}")


# --- MAIN --------------------------------------------------------------

def main(root_dir, overlap_threshold):
    bounds, shapes_dict = find_geotiff_files(root_dir)
    total = len(bounds)
    print(f"Total GeoTIFF patches discovered: {total}")
    if not total:
        return

    items = list(bounds.items())

    overlaps_csv = os.path.join(root_dir, "overlaps.csv")
    header = ["file1", "lon1", "lat1", "file2", "lon2", "lat2", "overlap_pct"]
    rows = []
    pair_list = []

    print(f"\nPairs with overlap ≥ {overlap_threshold:.3f}% "
          f"(intersection-over-min-area, sliver-filtered):\n")

    for i in range(total):
        f1, b1 = items[i]
        s1 = shapes_dict.get(f1)
        if s1 is None:
            continue
        for j in range(i + 1, total):
            f2, b2 = items[j]
            s2 = shapes_dict.get(f2)
            if s2 is None:
                continue

            # Fast bbox rejection
            if not boxes_overlap(b1, b2):
                continue

            pct = compute_overlap_pct(s1, s2)
            if pct < overlap_threshold:
                continue

            lon1, lat1 = centre(b1)
            lon2, lat2 = centre(b2)
            rows.append([
                os.path.basename(f1), f"{lon1:.6f}", f"{lat1:.6f}",
                os.path.basename(f2), f"{lon2:.6f}", f"{lat2:.6f}",
                f"{pct:.2f}",
            ])
            pair_list.append((f1, f2, pct))
            winner = random.choice(
                [os.path.basename(f1), os.path.basename(f2)]
            )
            print(
                f"  ↔ {os.path.basename(f1)} "
                f"({lon1:.6f},{lat1:.6f}) ↔ "
                f"{os.path.basename(f2)} "
                f"({lon2:.6f},{lat2:.6f}): "
                f"{pct:.2f}%  winner→ {winner}"
            )

    with open(overlaps_csv, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header)
        w.writerows(rows)
    print(f"\nLogged {len(rows)} overlapping pairs → {overlaps_csv}")

    if rows:
        kml_path = os.path.join(root_dir, "overlaps_all.kml")
        write_kml(pair_list, shapes_dict, kml_path)
    else:
        print("No overlaps ≥ threshold – no KML produced.")


main(os.path.abspath(root_dir), overlap_threshold)
print("✓ Done.")


Total GeoTIFF patches discovered: 10000

Pairs with overlap ≥ 0.000% (intersection-over-min-area, sliver-filtered):


Logged 0 overlapping pairs → /home/ubuntu/SENSERO/GeoTiff/Patch_336/overlaps.csv
No overlaps ≥ threshold – no KML produced.
✓ Done.
